# NeuroDevDiff — Hybrid Clinical Triage System

This notebook demonstrates a hybrid pipeline combining:

- DistilBERT classifier → defer / not defer decision
- LLM (Mistral) → rationale, clarifying questions, differential hypotheses

The system is designed for clinical decision support in developmental cases.

____________________________
# 1. Setup & load dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, os, gc
import torch
import tensorflow as tf
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score



## check path setup
# If you are running this notebook on Kaggle:
# Update the paths below according to your dataset location.
# Example:
# /kaggle/input/datasets/<your-username>/neurodevdiff

RUN_ENV = "kaggle"  # or "local"
if RUN_ENV == "kaggle":
    DATASET_DIR = Path("/kaggle/input/datasets/galvari/neurodevdiff")
    CLF_DIR = Path("/kaggle/input/datasets/galvari/distilbert-classifier/ndd-distilbert-classifier") 
else:
    PROJECT_ROOT = Path.cwd()
    DATASET_DIR = PROJECT_ROOT / "data"
    CLF_DIR = PROJECT_ROOT / "models/distilbert"


print("Dataset dir:", DATASET_DIR)
print("Classifier dir:", CLF_DIR)


files = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_file()])
print("Files:")
for f in files:
    print(" -", f)

csv_full = [f for f in files if f.endswith("_full.csv")]
if not csv_full:
    raise FileNotFoundError("Missing *_full.csv in dataset directory.")

prefix = csv_full[0].replace("_full.csv", "")
print("Detected prefix:", prefix)

df_full  = pd.read_csv(DATASET_DIR / f"{prefix}_full.csv")
df_train = pd.read_csv(DATASET_DIR / f"{prefix}_train.csv")
df_val   = pd.read_csv(DATASET_DIR / f"{prefix}_val.csv")
df_test  = pd.read_csv(DATASET_DIR / f"{prefix}_test.csv")

print("\nShapes:")
print("full :", df_full.shape)
print("train:", df_train.shape)
print("val  :", df_val.shape)
print("test :", df_test.shape)

train_jsonl = pd.read_json(DATASET_DIR / f"{prefix}_train.jsonl", lines=True)
val_jsonl   = pd.read_json(DATASET_DIR / f"{prefix}_val.jsonl", lines=True)
test_jsonl  = pd.read_json(DATASET_DIR / f"{prefix}_test.jsonl", lines=True)

print("\nJSONL shapes:")
print("train_jsonl:", train_jsonl.shape)
print("val_jsonl  :", val_jsonl.shape)
print("test_jsonl :", test_jsonl.shape)

with open(DATASET_DIR / f"{prefix}_metadata.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("\nMetadata keys:", list(meta.keys()))
print("Class balance:", meta.get("class_balance"))

def jsonl_to_xy(df_jsonl: pd.DataFrame):
    X = df_jsonl["input"].astype(str).values
    y = df_jsonl["output"].apply(lambda d: int(d["should_defer"])).values
    return X, y

X_train, y_train = jsonl_to_xy(train_jsonl)
X_val,   y_val   = jsonl_to_xy(val_jsonl)
X_test,  y_test  = jsonl_to_xy(test_jsonl)

print("Defer rate:")
print("train:", y_train.mean().round(3))
print("val  :", y_val.mean().round(3))
print("test :", y_test.mean().round(3))

print("\nSample vignette:\n", X_val[0][:400])
print("\nGT label:", y_val[0])

2026-03-23 12:23:14.596526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774268594.818843      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774268594.885056      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774268595.405435      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774268595.405480      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774268595.405483      55 computation_placer.cc:177] computation placer alr

Dataset dir: /kaggle/input/datasets/galvari/neurodevdiff
Classifier dir: /kaggle/input/datasets/galvari/distilbert-classifier/ndd-distilbert-classifier
Files:
 - neurodevdiff_v1_full.csv
 - neurodevdiff_v1_metadata.json
 - neurodevdiff_v1_test.csv
 - neurodevdiff_v1_test.jsonl
 - neurodevdiff_v1_train.csv
 - neurodevdiff_v1_train.jsonl
 - neurodevdiff_v1_val.csv
 - neurodevdiff_v1_val.jsonl
 - neurodevdiff_v1_vignettes.jsonl
Detected prefix: neurodevdiff_v1

Shapes:
full : (2000, 25)
train: (1399, 25)
val  : (300, 25)
test : (301, 25)

JSONL shapes:
train_jsonl: (1399, 6)
val_jsonl  : (300, 6)
test_jsonl : (301, 6)

Metadata keys: ['dataset', 'version', 'n_cases', 'seed', 'noise_level', 'created_utc', 'class_balance', 'defer_rate', 'risk_high_rate', 'config']
Class balance: {'ASD': 0.185, 'ADHD': 0.1685, 'ANXIETY': 0.1605, 'SLD': 0.145, 'OCD': 0.1065, 'SELECTIVE_MUTISM': 0.084, 'GDD_ID': 0.0815, 'NDD_UNSPEC': 0.069}
Defer rate:
train: 0.563
val  : 0.563
test : 0.568

Sample vignette:
 

____________________________
## 2. Load Classifier for defer decision

In [2]:
## load distilbert classifier and check

CLF_MODEL_DIR = CLF_DIR / "distilbert_defer_classifier"
CLF_TOKENIZER_DIR = CLF_DIR / "distilbert_tokenizer"
CLF_META_FILE = CLF_DIR / "distilbert_defer_meta.json"

print("MODEL:", CLF_MODEL_DIR)
print("TOKENIZER:", CLF_TOKENIZER_DIR)
print("META:", CLF_META_FILE)

print("Exists model:", CLF_MODEL_DIR.exists())
print("Exists tokenizer:", CLF_TOKENIZER_DIR.exists())
print("Exists meta:", CLF_META_FILE.exists())


# metadata
with open(CLF_META_FILE, "r") as f:
    clf_meta = json.load(f)

CLF_MAX_LEN = int(clf_meta["max_len"])
CLF_BEST_THR = float(clf_meta["best_threshold"])

print("MAX_LEN:", CLF_MAX_LEN)
print("BEST_THR:", CLF_BEST_THR)

# tokenizer
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_TOKENIZER_DIR)

# load savedmodel
clf_sm = tf.saved_model.load(str(CLF_MODEL_DIR))

print("Available signatures:", list(clf_sm.signatures.keys()))

infer = clf_sm.signatures["serving_default"]

print(infer.structured_input_signature)
print(infer.structured_outputs)



def encode_for_classifier(texts, tokenizer, max_len):
    enc = tokenizer(
        list(map(str, texts)),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="np"
    )
    return {
        "input_ids": tf.convert_to_tensor(enc["input_ids"], dtype=tf.int32),
        "attention_mask": tf.convert_to_tensor(enc["attention_mask"], dtype=tf.int32),
    }

def classifier_predict_one(vignette: str):
    x = encode_for_classifier([vignette], clf_tokenizer, CLF_MAX_LEN)

    out = infer(
        input_ids=x["input_ids"],
        attention_mask=x["attention_mask"]
    )

    out_tensor = out["dense_1"]  # dallo structured_outputs che hai stampato
    p_defer = float(tf.reshape(out_tensor, [-1]).numpy()[0])

    should_defer = int(p_defer >= CLF_BEST_THR)
    confidence = p_defer if should_defer == 1 else (1.0 - p_defer)

    return {
        "should_defer": should_defer,
        "defer_probability": p_defer,
        "confidence": confidence
    }



## test one sample
test_clf = classifier_predict_one(X_val[0])
print(json.dumps(test_clf, indent=2))
print("GT:", int(y_val[0]))


MODEL: /kaggle/input/datasets/galvari/distilbert-classifier/ndd-distilbert-classifier/distilbert_defer_classifier
TOKENIZER: /kaggle/input/datasets/galvari/distilbert-classifier/ndd-distilbert-classifier/distilbert_tokenizer
META: /kaggle/input/datasets/galvari/distilbert-classifier/ndd-distilbert-classifier/distilbert_defer_meta.json
Exists model: True
Exists tokenizer: True
Exists meta: True
MAX_LEN: 256
BEST_THR: 0.2


I0000 00:00:1774268631.116050      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774268631.118978      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Available signatures: ['serving_default']
((), {'attention_mask': TensorSpec(shape=(None, 256), dtype=tf.int32, name='attention_mask'), 'input_ids': TensorSpec(shape=(None, 256), dtype=tf.int32, name='input_ids')})
{'dense_1': TensorSpec(shape=(None, 1), dtype=tf.float32, name='dense_1')}
{
  "should_defer": 1,
  "defer_probability": 0.9831351041793823,
  "confidence": 0.9831351041793823
}
GT: 1


____________________________
## 3. Setup LLM 

In [3]:

tf.keras.backend.clear_session()
gc.collect()
torch.cuda.empty_cache()

LLM_ID = "mistralai/Mistral-7B-Instruct-v0.2"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

print("Loaded:", LLM_ID)


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.2


In [4]:

def generate_raw(prompt: str, max_new_tokens: int = 160) -> str:
    messages = [
        {"role": "system", "content": "You output only valid JSON."},
        {"role": "user", "content": prompt},
    ]

    chat_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(chat_text, return_tensors="pt").to(llm_model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=llm_tokenizer.eos_token_id,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][input_len:]
    text = llm_tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text.strip()


def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE).strip()

    m_obj = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m_obj:
        raise ValueError(f"No JSON object found in:\n{text[:500]}")

    return json.loads(m_obj.group(0))



def build_reasoning_prompt(vignette: str, clf_out: dict) -> str:
    return f"""A classifier already made the decision.

should_defer = {clf_out["should_defer"]}
confidence = {clf_out["confidence"]:.2f}

Do not change the decision.

Return ONLY one JSON object:
{{
  "rationale": "short decision-focused explanation",
  "clarifying_questions": ["question 1"],
  "differential_hypotheses": ["diagnosis 1"]
}}

Rules:
- rationale must be 1 or 2 short sentences
- rationale must explain only the main reason why the decision makes sense
- rationale must NOT summarize the case

- clarifying_questions must be 2 to 3 questions
- each question must target a critical missing piece of information
- each question must be useful to REDUCE uncertainty and potentially change the decision
- avoid generic or redundant questions

- differential_hypotheses: 1 to 3 plausible DSM-5 diagnosis labels
- no explanations in differential_hypotheses
- no text before or after JSON

CASE:
{vignette}
"""


____________________________
## 4. Hybrid Inference testing

In [5]:

def hybrid_predict(vignette: str):
    clf_out = classifier_predict_one(vignette)

    prompt = build_reasoning_prompt(vignette, clf_out)
    raw = generate_raw(prompt, max_new_tokens=250)

    try:
        parsed = extract_json(raw)
    except Exception:
        parsed = {
            "rationale": "",
            "clarifying_questions": [],
            "differential_hypotheses": []
        }

    final = {
        "should_defer": clf_out["should_defer"],
        "confidence": clf_out["confidence"],
        "defer_probability": clf_out["defer_probability"],
        "rationale": parsed.get("rationale", ""),
        "clarifying_questions": parsed.get("clarifying_questions", []),
        "differential_hypotheses": parsed.get("differential_hypotheses", [])
    }

    return final, raw


### testing
out, raw = hybrid_predict(X_val[0])

print("PARSED OUTPUT:\n")
print(json.dumps(out, indent=2))

print("\nRAW OUTPUT:\n")
print(raw[:1000])


PARSED OUTPUT:

{
  "should_defer": 1,
  "confidence": 0.9831351041793823,
  "defer_probability": 0.9831351041793823,
  "rationale": "The child's developmental and behavioral concerns, including checking, intrusive thoughts, counting, and avoidance of triggers, along with moderate severity and impact on daily routines, suggest a possible diagnosis of Obsessive-Compulsive Disorder (OCD).",
  "clarifying_questions": [
    "What is the exact onset timeline for the child's symptoms?",
    "Are there any tics or motor stereotypes present?"
  ],
  "differential_hypotheses": [
    "Obsessive-Compulsive Disorder (OCD)",
    "Tic Disorder",
    "Attention Deficit Hyperactivity Disorder (ADHD)"
  ]
}

RAW OUTPUT:

{
  "rationale": "The child's developmental and behavioral concerns, including checking, intrusive thoughts, counting, and avoidance of triggers, along with moderate severity and impact on daily routines, suggest a possible diagnosis of Obsessive-Compulsive Disorder (OCD).",
  "clarify

____________________________
## 5. Testing & Evaluation

In [6]:
rows = []

for i in range(100):
    out, raw = hybrid_predict(X_val[i])

    parse_ok = (
        len(str(out["rationale"]).strip()) > 0
        or len(out["questions"] if "questions" in out else out["clarifying_questions"]) > 0
        or len(out["diff"] if "diff" in out else out["differential_hypotheses"]) > 0
    )

    rows.append({
        "index": i,
        "gt": int(y_val[i]),
        "pred": out["should_defer"],
        "conf": out["confidence"],
        "rationale": out["rationale"],
        "questions": out["clarifying_questions"],
        "diff": out["differential_hypotheses"],
        "raw": raw,
        "parse_ok": parse_ok
    })

df_hybrid = pd.DataFrame(rows)

print('------------------- Classification Eval ---------------------')
print("Accuracy :", accuracy_score(df_hybrid["gt"], df_hybrid["pred"]))
print("Precision:", precision_score(df_hybrid["gt"], df_hybrid["pred"], zero_division=0))
print("Recall   :", recall_score(df_hybrid["gt"], df_hybrid["pred"], zero_division=0))
print("F1       :", f1_score(df_hybrid["gt"], df_hybrid["pred"], zero_division=0))
print("ROC-AUC  :", roc_auc_score(df_hybrid["gt"], df_hybrid["conf"]))

def llm_format_metrics(df):
    return {
        "n_cases": len(df),
        "parse_success_rate": df["parse_ok"].mean(),
        "nonempty_rationale_rate": (df["rationale"].astype(str).str.len() > 0).mean(),
        "questions_nonempty_rate": df["questions"].apply(lambda x: isinstance(x, list) and len(x) > 0).mean(),
        "diff_nonempty_rate": df["diff"].apply(lambda x: isinstance(x, list) and len(x) > 0).mean(),
        "avg_n_questions": df["questions"].apply(lambda x: len(x) if isinstance(x, list) else 0).mean(),
        "avg_n_diff": df["diff"].apply(lambda x: len(x) if isinstance(x, list) else 0).mean(),
    }


print('------------------- LLM-Output Eval ---------------------')
print(llm_format_metrics(df_hybrid))

df_hybrid.head()

------------------- Classification Eval ---------------------
Accuracy : 0.92
Precision: 1.0
Recall   : 0.8518518518518519
F1       : 0.92
ROC-AUC  : 0.8180354267310789
------------------- LLM-Output Eval ---------------------
{'n_cases': 100, 'parse_success_rate': np.float64(0.94), 'nonempty_rationale_rate': np.float64(0.94), 'questions_nonempty_rate': np.float64(0.94), 'diff_nonempty_rate': np.float64(0.94), 'avg_n_questions': np.float64(1.89), 'avg_n_diff': np.float64(2.69)}


,index,gt,pred,conf,rationale,questions,diff,raw,parse_ok
0,0,1,1,0.983135,The child's developmental and behavioral conce...,[What is the exact onset timeline for the chil...,"[Obsessive-Compulsive Disorder (OCD), Tic Diso...","{\n ""rationale"": ""The child's developmental a...",True
1,1,0,0,0.869873,The child's developmental and behavioral conce...,[What are the specific symptoms exhibited at h...,[Attention Deficit Hyperactivity Disorder (ADH...,"{\n ""rationale"": ""The child's developmental a...",True
2,2,1,1,0.998281,Moderate impairment in daily routines with cor...,[What is the extent of functional impairment i...,"[Attention Deficit Hyperactivity Disorder, Gen...","{\n ""rationale"": ""Moderate impairment in dail...",True
3,3,0,0,0.849440,Cognitive profile suggests a learning disorder...,[What are the specific results of the teacher ...,"[Specific Learning Disorder, Attention Deficit...","{\n ""rationale"": ""Cognitive profile suggests ...",True
4,4,1,1,0.998985,"The child's symptoms of restlessness, interrup...",[What are the specific details of the function...,"[Attention Deficit Hyperactivity Disorder, Com...","{\n ""rationale"": ""The child's symptoms of res...",True


In [18]:
## deep evaluation of LLM output

def question_diversity(df):
    all_q = [q for qs in df["questions"] if isinstance(qs, list) for q in qs]
    if len(all_q) == 0:
        return 0.0
    return len(set(all_q)) / len(all_q)

def rationale_length_stats(df):
    lengths = df["rationale"].astype(str).apply(lambda x: len(x.split()))
    return {
        "avg_rationale_words": lengths.mean(),
        "median_rationale_words": lengths.median()
    }



def build_judge_prompt(vignette: str, out: dict) -> str:
    return f"""You are a strict evaluator of clinical reasoning outputs.

CASE:
{vignette}

OUTPUT:
Rationale: {out.get("rationale", "")}
Clarifying questions: {out.get("questions", out.get("clarifying_questions", []))}
Differential hypotheses: {out.get("diff", out.get("differential_hypotheses", []))}

Evaluate from 1 to 5:
- rationale_specificity
- question_usefulness
- differential_plausibility

Rules:
- 1 = poor / generic
- 3 = moderate
- 5 = strong / specific
- All values must be integers between 1 and 5

overall_score must be the average of the three scores, rounded to nearest integer.

Return ONLY a valid JSON object.
Do NOT include any text before or after JSON.
Do NOT use markdown.

{{
  "rationale_specificity": 0,
  "question_usefulness": 0,
  "differential_plausibility": 0,
  "overall_score": 0,
  "short_reason": ""
}}
"""


def extract_judge_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE).strip()

    try:
        return json.loads(text)
    except:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        return json.loads(m.group(0))

    raise ValueError(f"No valid JSON found in:\n{text[:500]}")

def mistral_judge_case(vignette: str, out: dict, max_new_tokens: int = 180):
    prompt = build_judge_prompt(vignette, out)
    raw = generate_raw(prompt, max_new_tokens=max_new_tokens)
    parsed = extract_judge_json(raw)
    return parsed, raw


print('------------------- LLM Self-Eval ---------------------')
print({"question_diversity": question_diversity(df_hybrid)})
print(rationale_length_stats(df_hybrid))

sample_out = {
    "rationale": df_hybrid.iloc[0]["rationale"],
    "questions": df_hybrid.iloc[0]["questions"],
    "diff": df_hybrid.iloc[0]["diff"],
}

judge_parsed, judge_raw = mistral_judge_case(
    X_val[df_hybrid.iloc[0]["index"]],
    sample_out
)


judge_rows = []

for i in range(20):
    out_dict = {
        "rationale": df_hybrid.iloc[i]["rationale"],
        "questions": df_hybrid.iloc[i]["questions"],
        "diff": df_hybrid.iloc[i]["diff"],
    }

    try:
        judged, raw = mistral_judge_case(
            X_val[df_hybrid.iloc[i]["index"]],
            out_dict
        )

        judge_rows.append({
            "index": int(df_hybrid.iloc[i]["index"]),
            "rationale_specificity": judged.get("rationale_specificity", None),
            "question_usefulness": judged.get("question_usefulness", None),
            "differential_plausibility": judged.get("differential_plausibility", None),
            "overall_score": judged.get("overall_score", None),
            "short_reason": judged.get("short_reason", ""),
            "judge_parse_ok": True
        })

    except Exception:
        judge_rows.append({
            "index": int(df_hybrid.iloc[i]["index"]),
            "rationale_specificity": None,
            "question_usefulness": None,
            "differential_plausibility": None,
            "overall_score": None,
            "short_reason": "",
            "judge_parse_ok": False
        })

df_judge = pd.DataFrame(judge_rows)

print("Judge parse success rate:", df_judge["judge_parse_ok"].mean())
print(df_judge[
    ["rationale_specificity", "question_usefulness", "differential_plausibility", "overall_score"]
].mean(numeric_only=True))

df_judge.head()

------------------- LLM Self-Eval ---------------------
{'question_diversity': 0.8994708994708994}
{'avg_rationale_words': np.float64(26.78), 'median_rationale_words': 27.0}
Judge parse success rate: 0.9
rationale_specificity        2.666667
question_usefulness          4.444444
differential_plausibility    2.888889
overall_score                3.111111
dtype: float64


,index,rationale_specificity,question_usefulness,differential_plausibility,overall_score,short_reason,judge_parse_ok
0,0,3.0,5.0,3.0,4.0,The rationale is specific enough to suggest OC...,True
1,1,3.0,5.0,3.0,3.0,The rationale is somewhat specific as it menti...,True
2,2,3.0,5.0,3.0,3.0,Moderate impairment and core features suggest ...,True
3,3,3.0,5.0,3.0,4.0,The rationale is specific enough to suggest a ...,True
4,4,3.0,5.0,3.0,4.0,The rationale is specific enough to suggest AD...,True
